### Source Tables:
- `_exponent`.`_bronze_allscripts_scm_prod_01`.dbo_cv3client

### To Do:
-
### Notes:
- 

In [0]:
%sql
SELECT COUNT(DISTINCT GUID)
FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.dbo_cv3client
-- FROM _exponent._bronze_allscripts_tw_works_vw.dbo_person 
-- WHERE dbo_cv3client.



-- WHERE etl_load_ts BETWEEN  CURRENT_DATE() - INTERVAL 14 DAY AND CURRENT_DATE()




In [0]:
%sql
SELECT *
FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.dbo_cv3client
LIMIT 100;

In [0]:
# %sql
# SELECT GenderCode, COUNT(*) AS cnt
# FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.`dbo_cv3client`
# GROUP BY GenderCode
# ORDER BY cnt DESC

In [0]:
# %sql
# SELECT *
# from _exponent.omop_mapping.domain_source_to_concept gender_concept

In [0]:
# %sql
# SELECT RaceCode, COUNT(*) AS cnt
# FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.`dbo_cv3client`
# GROUP BY RaceCode
# ORDER BY cnt DESC

# Transformation

In [0]:
source = 'allscripts_scm'


In [0]:
# %sql
# SELECT * FROM _exponent.omop_mapping.domain_source_to_concept
# -- WHERE source_system = 'allscripts_scm'
# -- currently just tw data

In [0]:
source = 'allscripts_scm'

silver_person_df = spark.sql(f'''
SELECT 
  COALESCE(gender_concept.omop_concept_id, 0) AS gender_concept_id,
  c.BirthYearNum AS year_of_birth,
  c.BirthMonthNum AS month_of_birth,
  c.BirthDayNum AS day_of_birth,
  TRY_CAST(
    CONCAT(
      LPAD(CAST(c.BirthYearNum AS STRING), 4, '0'), '-',
      LPAD(CAST(c.BirthMonthNum AS STRING), 2, '0'), '-',
      LPAD(CAST(c.BirthDayNum AS STRING), 2, '0')
    ) AS TIMESTAMP
  ) AS birth_datetime,
  COALESCE(race_concept.omop_concept_id, 0) AS race_concept_id,
  0 AS ethnicity_concept_id, --=
  NULL AS location_id,--= location table 
  NULL AS provider_id, --
  NULL AS care_site_id, --
  CONCAT('{source}', ' | ', CAST(c.GUID AS STRING)) AS person_source_value,
  c.GenderCode AS gender_source_value,
  0 AS gender_source_concept_id, --=
  c.RaceCode AS race_source_value, 
  0 AS race_source_concept_id, --=
  NULL AS ethnicity_source_value,  --=
  0 AS ethnicity_source_concept_id, --=
  '{source}' AS source_system
FROM `_exponent`.`_bronze_allscripts_scm_prod_01`.`dbo_cv3client` c
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept gender_concept
  ON gender_concept.source_id = c.GenderCode
 AND gender_concept.domain_id = 'Gender'
 AND gender_concept.source_system = '{source}'
LEFT OUTER JOIN _exponent.omop_mapping.domain_source_to_concept race_concept
  ON race_concept.source_id = c.RaceCode
 AND race_concept.domain_id = 'Race'
 AND race_concept.source_system = '{source}'
WHERE c.GUID IS NOT NULL
''')

display(silver_person_df)
silver_person_df.createOrReplaceTempView("silver_person")

Update Write to Silver Logic for SCM

In [0]:
%sql
MERGE INTO _exponent.omop_silver.person AS t
USING silver_person AS s
ON t.person_source_value = s.person_source_value

WHEN MATCHED AND (
     NOT (t.gender_concept_id <=> s.gender_concept_id)
  OR NOT (t.year_of_birth <=> s.year_of_birth)
  OR NOT (t.month_of_birth <=> s.month_of_birth)
  OR NOT (t.day_of_birth <=> s.day_of_birth)
  OR NOT (t.birth_datetime <=> s.birth_datetime)
  OR NOT (t.race_concept_id <=> s.race_concept_id)
  OR NOT (t.ethnicity_concept_id <=> s.ethnicity_concept_id)
  OR NOT (t.location_id <=> s.location_id)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.care_site_id <=> s.care_site_id)
  -- OR NOT (t.person_source_value <=> s.person_source_value)
  OR NOT (t.gender_source_value <=> s.gender_source_value)
  OR NOT (t.gender_source_concept_id <=> s.gender_source_concept_id)
  OR NOT (t.race_source_value <=> s.race_source_value)
  OR NOT (t.race_source_concept_id <=> s.race_source_concept_id)
  OR NOT (t.ethnicity_source_value <=> s.ethnicity_source_value)
  OR NOT (t.ethnicity_source_concept_id <=> s.ethnicity_source_concept_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.gender_concept_id           = s.gender_concept_id,
  t.year_of_birth               = s.year_of_birth,
  t.month_of_birth              = s.month_of_birth,
  t.day_of_birth                = s.day_of_birth,
  t.birth_datetime              = s.birth_datetime,
  t.race_concept_id             = s.race_concept_id,
  t.ethnicity_concept_id        = s.ethnicity_concept_id,
  t.location_id                 = s.location_id,
  t.provider_id                 = s.provider_id,
  t.care_site_id                = s.care_site_id,
  -- t.person_source_value         = s.person_source_value,
  t.gender_source_value         = s.gender_source_value,
  t.gender_source_concept_id    = s.gender_source_concept_id,
  t.race_source_value           = s.race_source_value,
  t.race_source_concept_id      = s.race_source_concept_id,
  t.ethnicity_source_value      = s.ethnicity_source_value,
  t.ethnicity_source_concept_id = s.ethnicity_source_concept_id,
  t.source_system               = s.source_system,
  t.last_mod_tsp                = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
  -- person_id,
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  birth_datetime,
  race_concept_id,
  ethnicity_concept_id,
  location_id,
  provider_id,
  care_site_id,
  person_source_value,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id,
  source_system,
  last_mod_tsp
)
VALUES (
  s.gender_concept_id,
  s.year_of_birth,
  s.month_of_birth,
  s.day_of_birth,
  s.birth_datetime,
  s.race_concept_id,
  s.ethnicity_concept_id,
  s.location_id,
  s.provider_id,
  s.care_site_id,
  s.person_source_value,
  s.gender_source_value,
  s.gender_source_concept_id,
  s.race_source_value,
  s.race_source_concept_id,
  s.ethnicity_source_value,
  s.ethnicity_source_concept_id,
  s.source_system,
  current_timestamp()
);


In [0]:
%sql
SELECT * FROM _exponent.omop_silver.person

In [0]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_person (
    source_system,
    person_source_value,
    -- person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.person_source_value,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    COALESCE(s.last_mod_tsp, current_timestamp()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT source_system, person_source_value, last_mod_tsp
    FROM _exponent.omop_silver.person
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_person x
  ON s.person_source_value = x.person_source_value;

In [0]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_person 

In [0]:
%sql
MERGE INTO _exponent.omop.person AS gold_person
USING (
  SELECT
    source_to_person.person_id,                    
    s.gender_concept_id,
    s.year_of_birth,
    s.month_of_birth,
    s.day_of_birth,
    s.birth_datetime,
    s.race_concept_id,
    s.ethnicity_concept_id,
    s.location_id,
    s.provider_id,
    s.care_site_id,
    s.person_source_value,
    s.gender_source_value,
    s.gender_source_concept_id,
    s.race_source_value,
    s.race_source_concept_id,
    s.ethnicity_source_value,
    s.ethnicity_source_concept_id
  FROM _exponent.omop_silver.person s
  JOIN _exponent.omop_mapping.source_to_person
    ON source_to_person.person_source_value = s.person_source_value
   AND source_to_person.active_flag = TRUE
) AS src
ON gold_person.person_id = src.person_id

WHEN MATCHED THEN UPDATE SET
  gold_person.gender_concept_id = src.gender_concept_id,
  gold_person.year_of_birth = src.year_of_birth,
  gold_person.month_of_birth = src.month_of_birth,
  gold_person.day_of_birth = src.day_of_birth,
  gold_person.birth_datetime = src.birth_datetime,
  gold_person.race_concept_id = src.race_concept_id,
  gold_person.ethnicity_concept_id = src.ethnicity_concept_id,
  gold_person.location_id = src.location_id,
  gold_person.provider_id = src.provider_id,
  gold_person.care_site_id = src.care_site_id,
  gold_person.person_source_value = src.person_source_value,
  gold_person.gender_source_value = src.gender_source_value,
  gold_person.gender_source_concept_id = src.gender_source_concept_id,
  gold_person.race_source_value = src.race_source_value,
  gold_person.race_source_concept_id = src.race_source_concept_id,
  gold_person.ethnicity_source_value = src.ethnicity_source_value,
  gold_person.ethnicity_source_concept_id = src.ethnicity_source_concept_id

WHEN NOT MATCHED THEN INSERT (
  person_id,
  gender_concept_id,
  year_of_birth,
  month_of_birth,
  day_of_birth,
  birth_datetime,
  race_concept_id,
  ethnicity_concept_id,
  location_id,
  provider_id,
  care_site_id,
  person_source_value,
  gender_source_value,
  gender_source_concept_id,
  race_source_value,
  race_source_concept_id,
  ethnicity_source_value,
  ethnicity_source_concept_id
)
VALUES (
  src.person_id,
  src.gender_concept_id,
  src.year_of_birth,
  src.month_of_birth,
  src.day_of_birth,
  src.birth_datetime,
  src.race_concept_id,
  src.ethnicity_concept_id,
  src.location_id,
  src.provider_id,
  src.care_site_id,
  src.person_source_value,
  src.gender_source_value,
  src.gender_source_concept_id,
  src.race_source_value,
  src.race_source_concept_id,
  src.ethnicity_source_value,
  src.ethnicity_source_concept_id
);
